# Fine-tune Laya untuk gerbang PANDU

Notebook ini menjalankan langkah 4 dan 5 dari `api/eval/laya/README.md`:
melatih `laya-multilingual` dengan dataset hasil `bangun_dataset.py`, lalu
mengujinya pada 64 pesan berlabel dan membandingkannya dengan JEV.

**Sebelum Run All**
1. Unggah `api/eval/laya/data/kaggle.zip` (dibuat `bangun_dataset.py`) sebagai
   Kaggle Dataset privat. Isinya `latih_laya.py`, `evaluasi.py`, train, val, uji,
   dan manifest.
2. Di panel kanan notebook: *Add Input* → dataset tadi; *Accelerator* → **GPU T4 x1**
   (atau P100); *Internet* → **On** (untuk `pip install` dan unduhan model).
3. Run All. Perkiraan: 10–20 menit di satu T4 untuk ±2.100 pesan × 4 epoch.

Hasil: `/kaggle/working/laya-pandu-gerbang.tgz` (±600 MB) di tab *Output*.

In [ ]:
!nvidia-smi -L
!pip install -q "laya @ git+https://github.com/NandhaKishorM/laya@v0.3.22"
import laya
import torch
import transformers

print(
    "laya",
    laya.__version__ if hasattr(laya, "__version__") else "?",
    "| torch",
    torch.__version__,
    "| transformers",
    transformers.__version__,
    "| cuda",
    torch.cuda.is_available(),
)

In [ ]:
# Cari folder input yang berisi train.jsonl dan skrip, lalu salin ke /kaggle/working.
import glob
import os
import shutil
from pathlib import Path

train = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert train, (
    "train.jsonl tidak ditemukan: tambahkan Kaggle Dataset berisi file dari api/eval/laya/"
)
DATA = os.path.dirname(train[0])
KERJA = "/kaggle/working"
for nama in ("latih_laya.py", "evaluasi.py"):
    sumber = glob.glob(f"/kaggle/input/**/{nama}", recursive=True)
    assert sumber, f"{nama} tidak ditemukan di input"
    shutil.copy(sumber[0], KERJA)
os.makedirs(f"{KERJA}/data", exist_ok=True)
for nama in ("train.jsonl", "val.jsonl", "uji.jsonl", "manifest.json"):
    shutil.copy(os.path.join(DATA, nama), f"{KERJA}/data/")
print(Path(f"{KERJA}/data/manifest.json").read_text()[:1500])

## Latih

Loss dan hiperparameter mengikuti skrip resmi Laya (RLCD). Bobot terbaik dipilih
menurut cross-entropy validasi; temperature dikalibrasi pada set validasi.
Metrik per epoch: akurasi dan jumlah pesan akademik yang akan terblokir (ambang
produksi 0,7/0,9 pada probabilitas label).

In [ ]:
%cd /kaggle/working
!python latih_laya.py --data data --out laya-pandu-gerbang --epochs 4 --micro-batch 16

## Uji pada 64 pesan berlabel

Pembanding: checkpoint asli (zero-shot) dan angka JEV. Syarat lulus: 0/38 akademik
terblokir dan ≥ 8/9 pesan di luar topik terblokir. Keputusan blokir dihitung dengan
dua ukuran keyakinan: `confidence` (yang dibaca `gate.py` sekarang; pada Laya ini
entropi ternormalisasi) dan `answer_confidence` (probabilitas terkalibrasi).

In [ ]:
OUT = "laya-pandu-gerbang"
!python evaluasi.py --uji data/uji.jsonl --model $OUT --device cuda --out hasil_uji.jsonl

In [ ]:
!python evaluasi.py --uji data/uji.jsonl --model basis --device cuda | head -20

In [ ]:
!cat laya-pandu-gerbang/laporan_latih.json | head -80
!tar czf laya-pandu-gerbang.tgz laya-pandu-gerbang && ls -la laya-pandu-gerbang.tgz